# 3D Printing Log Analysis: 2025-2026

This notebook analyses the makerspace 3D printing log for the 2025-2026 year.

The basic framework is:

1. Load the TSV file.
2. Clean dates, times, numbers, and text columns.
3. Create useful analysis columns, such as month, week, turnaround days, print hours, and filament kilograms.
4. Build summary tables.
5. Draw graphs to show trends and patterns.

## 0. Install chart libraries if needed

Run the next cell only if your Jupyter notebook says `pandas`, `matplotlib`, or `seaborn` is missing, or if charts fail with a NumPy/Matplotlib error.

After installing packages, restart the Jupyter kernel, then run the notebook again from the top.

In [ ]:
# If packages are missing, remove the # from the next line and run this cell once.
# %pip install --upgrade pandas matplotlib seaborn

# If you see a NumPy/Matplotlib compatibility error, try this line instead, then restart the kernel.
# %pip install --upgrade "numpy<2" pandas matplotlib seaborn

## 1. Import libraries and load the file

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 11

DATA_FILE = Path("25-26 AutoQueue - Print Log.tsv")

# The first two rows in this file are dashboard summary rows.
# The real table header starts on row 3, so we skip the first two rows.
df = pd.read_csv(DATA_FILE, sep="	", skiprows=2)

# Remove accidental spaces around column names.
df.columns = df.columns.str.strip()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

## 2. Clean the data

A computer sees dates and times as text until we convert them. This section creates analysis-friendly columns.

In [ ]:
clean = df.copy()

# Dates are stored like 06/10/2025, so we parse them as day/month/year.
# Any non-date text becomes NaT, which means "not a time" or missing date.
clean["date_added"] = pd.to_datetime(clean["Date Added"], format="%d/%m/%Y", errors="coerce")
clean["date_completed"] = pd.to_datetime(clean["Date completed"], format="%d/%m/%Y", errors="coerce")

# Convert print duration text into hours.
clean["print_duration"] = pd.to_timedelta(clean["Print Time"], errors="coerce")
clean["time_taken_duration"] = pd.to_timedelta(clean["time taken for print"], errors="coerce")
clean["print_hours"] = clean["print_duration"].dt.total_seconds() / 3600
clean["time_taken_hours"] = clean["time_taken_duration"].dt.total_seconds() / 3600

# Convert filament from grams to both grams and kilograms.
clean["filament_g"] = pd.to_numeric(clean["Filament (g)"], errors="coerce")
clean["filament_kg"] = clean["filament_g"] / 1000

# Turnaround means how many calendar days passed between adding and completing the print.
clean["turnaround_days"] = (clean["date_completed"] - clean["date_added"]).dt.total_seconds() / 86400

# Time grouping columns.
clean["month"] = clean["date_added"].dt.to_period("M").astype(str)
clean["completion_month"] = clean["date_completed"].dt.to_period("M").astype(str).replace("NaT", np.nan)
clean["week"] = clean["date_added"].dt.strftime("%G-W%V")
clean["weekday"] = clean["date_added"].dt.day_name()

# Simple yes/no columns for easier counting.
clean["is_complete"] = clean["Status"].eq("Complete")
clean["is_failed"] = clean["Status"].eq("Failed")
clean["is_rejected"] = clean["Status"].eq("Rejected")

# Requeue is recorded in the Notes text. This counts one requeue per row with matching text.
clean["is_requeue"] = clean["Notes"].fillna("").str.contains(
    r"re[- ]?queued|requeue",
    case=False,
    regex=True,
)

# Some printer rows are blank. Give them a label so they do not disappear from charts.
clean["Printer"] = clean["Printer"].fillna("Unknown printer").replace("", "Unknown printer")
clean["Project type"] = clean["Project type"].fillna("Unknown project").replace("", "Unknown project")

# Some physical printers should be reported together because they belong to the same analysis group.
# The original Printer column stays unchanged; printer_group is the merged name used in printer charts.
printer_merge_map = {
    "Lamarr": "Bell",
    "Curie": "Tolstoy",
    "Davinci": "Einstein",
    "Lovelace": "Socrates",
}

clean["printer_group"] = clean["Printer"].replace(printer_merge_map)

clean[["Name", "Status", "Project type", "Printer", "printer_group", "is_requeue", "date_added", "date_completed", "completion_month", "turnaround_days", "print_hours", "filament_g"]].head()

## 3. Quick overview numbers

These are KPI numbers. KPI means key performance indicator: a single number that summarises something important.

In [ ]:
completed = clean[clean["is_complete"]].copy()

summary = pd.Series({
    "first_date_added": clean["date_added"].min().date(),
    "last_date_added": clean["date_added"].max().date(),
    "total_print_records": len(clean),
    "completed_prints": int(clean["is_complete"].sum()),
    "failed_prints": int(clean["is_failed"].sum()),
    "rejected_prints": int(clean["is_rejected"].sum()),
    "success_rate_percent": round(clean["is_complete"].mean() * 100, 1),
    "unique_users": clean["Email"].nunique(),
    "total_print_hours": round(clean["print_hours"].sum(), 1),
    "total_filament_kg": round(clean["filament_kg"].sum(), 1),
    "average_turnaround_days_completed": round(completed["turnaround_days"].mean(), 2),
    "median_turnaround_days_completed": round(completed["turnaround_days"].median(), 2),
})

summary

## 4. Monthly summary table

In [ ]:
monthly = (
    clean.groupby("month")
    .agg(
        prints=("Status", "size"),
        completed=("is_complete", "sum"),
        failed=("is_failed", "sum"),
        rejected=("is_rejected", "sum"),
        unique_users=("Email", "nunique"),
        total_print_hours=("print_hours", "sum"),
        total_filament_kg=("filament_kg", "sum"),
        avg_turnaround_days=("turnaround_days", "mean"),
        median_turnaround_days=("turnaround_days", "median"),
    )
    .reset_index()
)

monthly["success_rate_percent"] = monthly["completed"] / monthly["prints"] * 100
monthly = monthly.round({
    "total_print_hours": 1,
    "total_filament_kg": 1,
    "avg_turnaround_days": 2,
    "median_turnaround_days": 2,
    "success_rate_percent": 1,
})

monthly

## 5. Graph: prints per month

A bar chart is good for comparing monthly demand.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=monthly, x="month", y="prints", color="#4C78A8", ax=ax)
ax.set_title("Number of Print Records per Month")
ax.set_xlabel("Month")
ax.set_ylabel("Number of print records")
ax.bar_label(ax.containers[0], padding=3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Graph: average and median turnaround time

A line chart is good for seeing whether waiting time is getting better or worse. Median is useful because one very late print can pull the average up.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly, x="month", y="avg_turnaround_days", marker="o", label="Average", ax=ax)
sns.lineplot(data=monthly, x="month", y="median_turnaround_days", marker="o", label="Median", ax=ax)
ax.set_title("Turnaround Time per Month")
ax.set_xlabel("Month")
ax.set_ylabel("Turnaround time, days")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Graph: weekly demand and weekly turnaround

Weekly charts show short busy periods that monthly charts can hide.

In [ ]:
weekly = (
    clean.groupby("week")
    .agg(
        prints=("Status", "size"),
        avg_turnaround_days=("turnaround_days", "mean"),
        median_turnaround_days=("turnaround_days", "median"),
    )
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

sns.barplot(data=weekly, x="week", y="prints", color="#72B7B2", ax=ax1)
sns.lineplot(data=weekly, x="week", y="avg_turnaround_days", color="#E45756", marker="o", ax=ax2)

ax1.set_title("Weekly Print Demand and Average Turnaround")
ax1.set_xlabel("Week")
ax1.set_ylabel("Number of print records")
ax2.set_ylabel("Average turnaround, days")
ax1.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

weekly.sort_values("prints", ascending=False).head(10)

## 8. Graph: status mix by month

A stacked bar chart shows how many prints were complete, failed, rejected, or still queued.

In [ ]:
status_month = pd.crosstab(clean["month"], clean["Status"])

ax = status_month.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="Set2")
ax.set_title("Print Status by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Number of print records")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Status")
plt.tight_layout()
plt.show()

status_month

## 9. Graph: failure rate by printer group

This chart uses the merged printer groups. It still includes failed prints, because failure rate needs both successful and failed records.

In [ ]:
printer_summary = (
    clean.groupby("printer_group")
    .agg(
        prints=("Status", "size"),
        completed=("is_complete", "sum"),
        failed=("is_failed", "sum"),
        total_print_hours=("print_hours", "sum"),
        total_filament_kg=("filament_kg", "sum"),
        avg_turnaround_days=("turnaround_days", "mean"),
    )
    .reset_index()
)

printer_summary["failure_rate_percent"] = printer_summary["failed"] / printer_summary["prints"] * 100
printer_summary = printer_summary.round({
    "total_print_hours": 1,
    "total_filament_kg": 1,
    "avg_turnaround_days": 2,
    "failure_rate_percent": 1,
})

printer_failure = printer_summary[printer_summary["prints"] >= 20].sort_values("failure_rate_percent", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=printer_failure, y="printer_group", x="failure_rate_percent", color="#F58518", ax=ax)
ax.set_title("Failure Rate by Printer Group")
ax.set_xlabel("Failure rate (%)")
ax.set_ylabel("Printer group")
plt.tight_layout()
plt.show()

printer_failure[["printer_group", "prints", "failed", "failure_rate_percent"]]

## 10. Graph: most used printers - completed prints only

This chart counts only completed prints. It also uses the merged printer groups, so Lamarr is counted with Bell, Curie with Tolstoy, and Davinci with Einstein.

In [ ]:
completed_for_printer_usage = clean[clean["is_complete"]].copy()

printer_usage_summary = (
    completed_for_printer_usage.groupby("printer_group")
    .agg(
        prints=("Status", "size"),
        total_print_hours=("print_hours", "sum"),
        total_filament_kg=("filament_kg", "sum"),
        avg_turnaround_days=("turnaround_days", "mean"),
    )
    .reset_index()
)

printer_usage_summary = printer_usage_summary.round({
    "total_print_hours": 1,
    "total_filament_kg": 1,
    "avg_turnaround_days": 2,
})

top_printers = printer_usage_summary.sort_values("prints", ascending=False).head(12)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_printers, y="printer_group", x="prints", color="#54A24B", ax=ax)
ax.set_title("Most Used Printers - Completed Prints Only")
ax.set_xlabel("Number of completed prints")
ax.set_ylabel("Printer group")
plt.tight_layout()
plt.show()

top_printers[["printer_group", "prints", "total_print_hours", "total_filament_kg", "avg_turnaround_days"]]

## 11. Graph: project type demand

This shows which types of work are using the makerspace most.

In [ ]:
project_summary = (
    clean.groupby("Project type")
    .agg(
        prints=("Status", "size"),
        total_print_hours=("print_hours", "sum"),
        total_filament_kg=("filament_kg", "sum"),
        avg_turnaround_days=("turnaround_days", "mean"),
    )
    .reset_index()
    .sort_values("prints", ascending=False)
)

top_projects = project_summary.head(12)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_projects, y="Project type", x="prints", color="#B279A2", ax=ax)
ax.set_title("Top Project Types by Number of Prints")
ax.set_xlabel("Number of print records")
ax.set_ylabel("Project type")
plt.tight_layout()
plt.show()

top_projects.round(2)

## 12. Graph: print hours and filament used by month

This combines two resource measures in one graph. Bars show machine time, and the line shows material use.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

bars = sns.barplot(data=monthly, x="month", y="total_print_hours", color="#4C78A8", ax=ax1)
line = sns.lineplot(data=monthly, x="month", y="total_filament_kg", color="#F58518", marker="o", linewidth=2.5, ax=ax2)

ax1.set_title("Monthly Print Hours and Filament Used")
ax1.set_xlabel("Month")
ax1.set_ylabel("Total print hours")
ax2.set_ylabel("Total filament used, kg")
ax1.tick_params(axis="x", rotation=45)

ax1.bar_label(ax1.containers[0], fmt="%.0f", padding=3)

# Make one combined legend, even though the bars and line use different y-axes.
bar_proxy = plt.Line2D([0], [0], color="#4C78A8", linewidth=8)
line_proxy = plt.Line2D([0], [0], color="#F58518", marker="o", linewidth=2.5)
ax1.legend([bar_proxy, line_proxy], ["Print hours", "Filament kg"], loc="upper left")

plt.tight_layout()
plt.show()

## 13. Graph: turnaround distribution with normal curve

A histogram shows the real waiting-time pattern. The bell curve is a fitted normal distribution, which gives a useful reference, but the actual data may still be skewed by unusually delayed prints.

In [ ]:
turnaround = completed["turnaround_days"].dropna()
turnaround_mean = turnaround.mean()
turnaround_std = turnaround.std(ddof=1)

x_min = max(0, turnaround.min())
x_max = max(turnaround.max(), turnaround_mean + 4 * turnaround_std)
x_values = np.linspace(x_min, x_max, 400)
normal_pdf = (1 / (turnaround_std * np.sqrt(2 * np.pi))) * np.exp(
    -0.5 * ((x_values - turnaround_mean) / turnaround_std) ** 2
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(turnaround, bins=30, stat="density", color="#4C78A8", alpha=0.55, ax=axes[0])
axes[0].plot(x_values, normal_pdf, color="#E45756", linewidth=2.5, label="Fitted normal curve")
axes[0].axvline(turnaround_mean, color="#222222", linestyle="--", linewidth=1.5, label=f"Mean: {turnaround_mean:.2f} days")
axes[0].axvspan(max(0, turnaround_mean - turnaround_std), turnaround_mean + turnaround_std, color="#72B7B2", alpha=0.15, label="Approx. 68% range")
axes[0].axvspan(max(0, turnaround_mean - 2 * turnaround_std), turnaround_mean + 2 * turnaround_std, color="#F58518", alpha=0.08, label="Approx. 95% range")
axes[0].set_title("Turnaround Time Distribution with Fitted Normal Curve")
axes[0].set_xlabel("Turnaround time, days")
axes[0].set_ylabel("Density")
axes[0].legend()

sns.boxplot(x=turnaround, color="#72B7B2", ax=axes[1])
axes[1].set_title("Turnaround Time Box Plot")
axes[1].set_xlabel("Turnaround time, days")

plt.tight_layout()
plt.show()

turnaround_stats = pd.Series({
    "completed_prints_with_turnaround": int(turnaround.count()),
    "mean_days": round(turnaround_mean, 2),
    "standard_deviation_days": round(turnaround_std, 2),
    "median_days": round(turnaround.median(), 2),
    "approx_68_percent_range": f"{max(0, turnaround_mean - turnaround_std):.2f} to {turnaround_mean + turnaround_std:.2f} days",
    "approx_95_percent_range": f"{max(0, turnaround_mean - 2 * turnaround_std):.2f} to {turnaround_mean + 2 * turnaround_std:.2f} days",
    "max_days": round(turnaround.max(), 2),
})

turnaround_stats

## 14. Graph: printer usage heatmap

A heatmap is useful for spotting busy months for each printer group. This version counts only completed prints and places them in the month when they were completed.

In [ ]:
top_printer_names = top_printers["printer_group"].tolist()

printer_month = (
    completed_for_printer_usage[
        completed_for_printer_usage["printer_group"].isin(top_printer_names)
        & completed_for_printer_usage["completion_month"].notna()
    ]
    .pivot_table(index="printer_group", columns="completion_month", values="Status", aggfunc="count", fill_value=0)
)

plt.figure(figsize=(13, 7))
sns.heatmap(printer_month, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Completed Printer Usage Heatmap by Completion Month")
plt.xlabel("Completion month")
plt.ylabel("Printer group")
plt.tight_layout()
plt.show()

## 15. Fun graph: most common words in print filenames

This looks at the `Gcode Filename` column. It removes common technical words, numbers, and file extensions so the chart is more about what people printed.

In [ ]:
filename_text = clean["Gcode Filename"].fillna("").str.lower()
filename_text = filename_text.str.replace(r"\.[a-z0-9]+$", " ", regex=True)
filename_text = filename_text.str.replace(r"[_\/\-]+", " ", regex=True)
filename_text = filename_text.str.replace(r"[^a-z\s]+", " ", regex=True)

filename_stop_words = {
    "gcode", "print", "printed", "plate", "pla", "petg", "abs", "coreone",
    "prusa", "mk", "input", "shaper", "inputshaper", "mm", "min", "mins",
    "hour", "hours", "hr", "hrs", "draft", "final", "copy", "version",
    "left", "right", "front", "back", "top", "bottom", "part", "body",
    "with", "for", "and", "the", "new", "old", "ifprusa", "iforge",
}

words = []
for filename in filename_text:
    for word in filename.split():
        if len(word) >= 3 and word not in filename_stop_words:
            words.append(word)

filename_words = pd.DataFrame(Counter(words).most_common(20), columns=["word", "count"])

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=filename_words, y="word", x="count", color="#B279A2", ax=ax)
ax.set_title("Most Common Words in Print File Names")
ax.set_xlabel("Number of times word appears")
ax.set_ylabel("Word")
plt.tight_layout()
plt.show()

filename_words

## 16. Graph: requeues by printer group

This counts one requeue for each row where the `Notes` column contains `re-queued`, `requeued`, or `requeue`. It uses printer groups, so merged printers are counted together.

In [ ]:
requeue_by_printer = (
    clean.groupby("printer_group")
    .agg(
        requeues=("is_requeue", "sum"),
        total_records=("Status", "size"),
        completed=("is_complete", "sum"),
        failed=("is_failed", "sum"),
    )
    .reset_index()
)

requeue_by_printer["requeue_rate_percent"] = requeue_by_printer["requeues"] / requeue_by_printer["total_records"] * 100
requeue_by_printer = requeue_by_printer.sort_values("requeues", ascending=False).round({"requeue_rate_percent": 1})

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=requeue_by_printer, y="printer_group", x="requeues", color="#E45756", ax=ax)
ax.set_title("Notes-Based Requeues by Printer Group")
ax.set_xlabel("Number of rows marked as requeued in Notes")
ax.set_ylabel("Printer group")
plt.tight_layout()
plt.show()

requeue_by_printer

## 17. Graph: turnaround time by printer group

This uses completed prints with valid added and completed dates. Average turnaround is useful for comparison, while the table also shows the median so one unusually late job does not hide the typical wait.

In [ ]:
turnaround_by_printer = (
    completed_for_printer_usage.dropna(subset=["turnaround_days"])
    .groupby("printer_group")
    .agg(
        completed_with_dates=("Status", "size"),
        avg_turnaround_days=("turnaround_days", "mean"),
        median_turnaround_days=("turnaround_days", "median"),
        max_turnaround_days=("turnaround_days", "max"),
    )
    .reset_index()
    .sort_values("avg_turnaround_days", ascending=False)
)

turnaround_by_printer = turnaround_by_printer.round({
    "avg_turnaround_days": 2,
    "median_turnaround_days": 2,
    "max_turnaround_days": 2,
})

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=turnaround_by_printer, y="printer_group", x="avg_turnaround_days", color="#4C78A8", ax=ax)
ax.set_title("Average Turnaround Time by Printer Group")
ax.set_xlabel("Average turnaround time, days")
ax.set_ylabel("Printer group")
plt.tight_layout()
plt.show()

turnaround_by_printer

## 18. Graph: average print time by printer group

This uses completed jobs only. It shows how long a typical successful job takes on each printer group.

In [ ]:
avg_print_time_by_printer = (
    completed_for_printer_usage.dropna(subset=["print_hours"])
    .groupby("printer_group")
    .agg(
        completed_jobs=("Status", "size"),
        avg_print_hours=("print_hours", "mean"),
        median_print_hours=("print_hours", "median"),
        total_print_hours=("print_hours", "sum"),
    )
    .reset_index()
    .sort_values("avg_print_hours", ascending=False)
)

avg_print_time_by_printer = avg_print_time_by_printer.round({
    "avg_print_hours": 2,
    "median_print_hours": 2,
    "total_print_hours": 1,
})

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=avg_print_time_by_printer, y="printer_group", x="avg_print_hours", color="#54A24B", ax=ax)
ax.set_title("Average Print Time by Printer Group - Completed Jobs Only")
ax.set_xlabel("Average print time, hours")
ax.set_ylabel("Printer group")
plt.tight_layout()
plt.show()

avg_print_time_by_printer

## 19. Graph: print time vs filament

A scatter plot shows whether longer prints usually use more filament.

In [ ]:
scatter_data = clean.dropna(subset=["print_hours", "filament_g"])

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=scatter_data,
    x="print_hours",
    y="filament_g",
    hue="Status",
    alpha=0.7,
    ax=ax,
)
ax.set_title("Print Time vs Filament Used")
ax.set_xlabel("Print time, hours")
ax.set_ylabel("Filament, grams")
plt.tight_layout()
plt.show()

## 20. Graph: unique users per month

This chart shows whether the makerspace is serving more or fewer individual users over time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly, x="month", y="unique_users", marker="o", color="#E45756", ax=ax)
ax.set_title("Unique Users per Month")
ax.set_xlabel("Month")
ax.set_ylabel("Number of unique users")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 21. Optional: save cleaned data and summary tables

This creates CSV files that you can open in Excel. Run it only if you want exported tables.

In [ ]:
# Optional exports. Remove the # from each line if you want to save these files.
# clean.to_csv("25_26_cleaned_print_log.csv", index=False)
# monthly.to_csv("25_26_monthly_summary.csv", index=False)
# weekly.to_csv("25_26_weekly_summary.csv", index=False)
# printer_summary.to_csv("25_26_printer_summary.csv", index=False)
# project_summary.to_csv("25_26_project_summary.csv", index=False)
# filename_words.to_csv("25_26_filename_word_counts.csv", index=False)
# requeue_by_printer.to_csv("25_26_requeue_by_printer.csv", index=False)
# turnaround_by_printer.to_csv("25_26_turnaround_by_printer.csv", index=False)
# avg_print_time_by_printer.to_csv("25_26_avg_print_time_by_printer.csv", index=False)